# 데이터 품질 점검 & 제거 대상 탐지

In [ ]:
# ============================================================
# RFMiD Preprocessed 데이터 점검 파이프라인 (Colab)
# 1) 결함 데이터 탐지
# 2) 품질 낮은 이미지 제외 후보
# 3) 중복 데이터 탐지 (exact + near-duplicate)
# ============================================================

import os, re, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageStat
from tqdm import tqdm


In [ ]:

# -------------------------------
# 0) (필요 시) Google Drive 마운트
# -------------------------------
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception:
    pass

# ============================================================
# A. 설정 (전처리 노트북 구조 기준 기본값)
# ============================================================
OUTPUT_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Preprocessed"  # preprocessed split 폴더 (train/val/test)
REPORT_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports"       # 보고서 저장 폴더
CSV_DIR_DEFAULT = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv"           # 라벨 CSV가 풀려 있는 위치(환경에 맞게 수정 가능)

SPLITS = ["train", "val", "test"]
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# 라벨 CSV (존재하는 것만 사용됨)
LABEL_CSV_MAP = {
    "train": f"{CSV_DIR_DEFAULT}/Training_Labels.csv",
    "val":   f"{CSV_DIR_DEFAULT}/Validation_Labels.csv",
    "test":  f"{CSV_DIR_DEFAULT}/Testing_Labels.csv",
}

# 결함 기준(해상도)
MIN_W, MIN_H = 128, 128
MAX_W, MAX_H = 6000, 6000

# 품질 후보 기준(너무 타이트하면 후보가 과하게 나올 수 있어, 필요 시 조절)
BLUR_LAPL_VAR_TH = 60.0       # 작을수록 흐림(blur) 의심
LOW_CONTRAST_STD_TH = 18.0    # RGB 평균 표준편차가 작으면 대비 낮음 의심
DARK_MEAN_TH = 35.0           # 너무 어두움(underexposure) 의심
BRIGHT_MEAN_TH = 220.0        # 너무 밝음(overexposure) 의심
DARK_BORDER_RATIO_TH = 0.40   # 테두리(검은 영역) 비율이 너무 크면 FOV 부족/크롭 문제 의심

# 중복 탐지
DO_EXACT_DUP = True           # 파일 바이트(md5) 기반 완전 중복
DO_NEAR_DUP = True            # perceptual hash(dHash) 기반 유사 중복
DHASH_SIZE = 8                # 8이면 64-bit hash
NEAR_DUP_HAMMING_TH = 5       # 해밍거리 <= 5 이면 유사 중복 후보(조절 가능)

# 격리 옵션(원본 삭제 대신 quarantine 폴더로 이동)
ENABLE_QUARANTINE_MOVE = False
QUARANTINE_DIR = f"{REPORT_BASE_DIR}/quarantine"  # 이동 위치

# 결과 저장 폴더
RUN_TAG = "dataset_qc_run"
OUT_DIR = f"{REPORT_BASE_DIR}/{RUN_TAG}"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(QUARANTINE_DIR, exist_ok=True)

# ============================================================
# B. 유틸 함수
# ============================================================
def list_images(dir_path: str):
    p = Path(dir_path)
    if not p.exists():
        return []
    return [x for x in p.iterdir() if x.is_file() and x.suffix.lower() in IMG_EXTS]

def safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)

def quarantine_move(src_path: Path, split: str, reason: str):
    """원본을 quarantine로 이동 (옵션)"""
    dest_dir = Path(QUARANTINE_DIR) / split / reason
    safe_mkdir(str(dest_dir))
    dest = dest_dir / src_path.name
    if dest.exists():
        dest = dest_dir / f"{src_path.stem}__dup{src_path.suffix}"
    shutil.move(str(src_path), str(dest))
    return str(dest)

def find_id_col(df: pd.DataFrame) -> str:
    for c in ["ID", "id", "image_id", "ImageID", "img_id", "filename"]:
        if c in df.columns:
            return c
    return df.columns[0]

def find_dr_col(df: pd.DataFrame) -> str:
    if "DR" in df.columns:
        return "DR"
    for c in df.columns:
        if str(c).strip().lower() == "dr":
            return c
    raise ValueError(f"DR 컬럼을 찾지 못했어. columns={list(df.columns)}")

def load_label_map(csv_path: str):
    """
    RFMiD CSV -> {img_id(str): dr_bin(int)} 반환
    - binary용 DR만 사용
    """
    df = pd.read_csv(csv_path)
    id_col = find_id_col(df)
    dr_col = find_dr_col(df)
    df[id_col] = df[id_col].astype(str).str.strip()

    def to_bin(x):
        if pd.isna(x): return 0
        if isinstance(x, str):
            x2 = x.strip().lower()
            if x2 in ["1", "true", "yes", "y"]: return 1
            if x2 in ["0", "false", "no", "n"]: return 0
        try:
            return 1 if float(x) >= 1 else 0
        except:
            return 0

    df["DR_BIN"] = df[dr_col].apply(to_bin).astype(int)
    return dict(zip(df[id_col], df["DR_BIN"])), id_col, dr_col, int(df["DR_BIN"].sum()), len(df)

def pil_open_verify(path: Path):
    """깨진 이미지 탐지 + 메타 추출"""
    with Image.open(path) as img:
        img.verify()
    img = Image.open(path)
    return img

def rgb_image(img: Image.Image) -> Image.Image:
    """채널 통일: RGB로 변환"""
    if img.mode != "RGB":
        return img.convert("RGB")
    return img

def laplacian_var(gray_np: np.ndarray) -> float:
    """라플라시안 분산(blur 지표). OpenCV 없이 구현(간단 커널)"""
    # 3x3 Laplacian kernel
    k = np.array([[0, 1, 0],
                  [1,-4, 1],
                  [0, 1, 0]], dtype=np.float32)
    # conv (valid)
    g = gray_np.astype(np.float32)
    h, w = g.shape
    if h < 3 or w < 3:
        return 0.0
    out = (
        k[0,1]*g[0:h-2,1:w-1] +
        k[1,0]*g[1:h-1,0:w-2] +
        k[1,1]*g[1:h-1,1:w-1] +
        k[1,2]*g[1:h-1,2:w]   +
        k[2,1]*g[2:h,  1:w-1]
    )
    return float(out.var())

def dark_border_ratio(rgb_np: np.ndarray, thr: int = 15) -> float:
    """
    테두리(검은 영역) 비율 추정:
    - 전체 픽셀 중 '매우 어두운 픽셀' 비율을 사용(간단 휴리스틱)
    - 안저영상에서 FOV 밖 검은 배경이 지나치게 크면 후보로 잡힘
    """
    gray = rgb_np.mean(axis=2)
    return float((gray < thr).mean())

def file_md5(path: Path) -> str:
    m = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            m.update(chunk)
    return m.hexdigest()

def dhash(img: Image.Image, hash_size: int = 8) -> str:
    """
    dHash (perceptual hash): 64-bit hex string
    """
    img = img.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
    pixels = np.array(img)
    diff = pixels[:, 1:] > pixels[:, :-1]
    # bit -> int
    bit_string = ''.join('1' if b else '0' for b in diff.flatten())
    return hex(int(bit_string, 2))[2:].rjust(hash_size * hash_size // 4, '0')

def hamming_hex(a: str, b: str) -> int:
    return bin(int(a, 16) ^ int(b, 16)).count("1")

# ============================================================
# C. 1) 결함 데이터 탐지
#    - 깨진 파일(open/verify 실패)
#    - 해상도 이상치
#    - 채널 모드 기록
#    - 라벨 누락(라벨 CSV 기반)
# ============================================================
def step1_defect_scan():
    defects = []
    per_split_summary = []

    label_maps = {}
    for split in SPLITS:
        csv_path = LABEL_CSV_MAP.get(split)
        if csv_path and os.path.exists(csv_path):
            lm, id_col, dr_col, pos, total = load_label_map(csv_path)
            label_maps[split] = lm
            print(f"[labels:{split}] loaded {total} rows | pos(DR=1)={pos} | id_col={id_col} dr_col={dr_col}")
        else:
            label_maps[split] = None
            print(f"[labels:{split}] label csv not found -> 라벨 누락 탐지 스킵: {csv_path}")

    for split in SPLITS:
        split_dir = f"{OUTPUT_BASE_DIR}/{split}"
        imgs = list_images(split_dir)
        lm = label_maps.get(split)
        ok_cnt = 0

        for p in tqdm(imgs, desc=f"[STEP1] Defect scan {split}"):
            img_id = p.stem

            # 라벨 누락 체크(가능할 때만)
            if lm is not None and img_id not in lm:
                defects.append({
                    "split": split, "path": str(p), "img_id": img_id,
                    "defect_type": "missing_label", "detail": ""
                })
                if ENABLE_QUARANTINE_MOVE:
                    quarantine_move(p, split, "missing_label")
                continue

            # open/verify + 메타 체크
            try:
                img = pil_open_verify(p)
                w, h = img.size
                mode = img.mode

                if w < MIN_W or h < MIN_H:
                    defects.append({"split": split, "path": str(p), "img_id": img_id,
                                    "defect_type": "too_small", "detail": f"{w}x{h} mode={mode}"})
                    img.close()
                    if ENABLE_QUARANTINE_MOVE:
                        quarantine_move(p, split, "too_small")
                    continue

                if w > MAX_W or h > MAX_H:
                    defects.append({"split": split, "path": str(p), "img_id": img_id,
                                    "defect_type": "too_large", "detail": f"{w}x{h} mode={mode}"})
                    img.close()
                    if ENABLE_QUARANTINE_MOVE:
                        quarantine_move(p, split, "too_large")
                    continue

                # 특이 mode 기록(결함은 아님)
                if mode not in ["RGB", "L"]:
                    defects.append({"split": split, "path": str(p), "img_id": img_id,
                                    "defect_type": "unusual_mode", "detail": f"mode={mode} size={w}x{h}"})

                ok_cnt += 1
                img.close()

            except Exception as e:
                defects.append({"split": split, "path": str(p), "img_id": img_id,
                                "defect_type": "broken_open_fail", "detail": str(e)})
                if ENABLE_QUARANTINE_MOVE:
                    quarantine_move(p, split, "broken_open_fail")
                continue

        per_split_summary.append({"split": split, "images": len(imgs), "ok(rough)": ok_cnt})

    # ✅ 결함 0건이어도 컬럼 유지
    defects_cols = ["split", "path", "img_id", "defect_type", "detail"]
    defects_df = pd.DataFrame(defects, columns=defects_cols)
    if len(defects_df) > 0:
        defects_df = defects_df.sort_values(["split", "defect_type", "img_id"], ascending=True)

    summary_df = pd.DataFrame(per_split_summary)

    defects_path = f"{OUT_DIR}/step1_defects.csv"
    summary_path = f"{OUT_DIR}/step1_summary.csv"
    defects_df.to_csv(defects_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    print("\n✅ STEP1 저장 완료")
    print(" - defects:", defects_path, f"(rows={len(defects_df)})")
    print(" - summary:", summary_path)

    return defects_df

# ============================================================
# D. 2) 품질 낮은 이미지 제외 후보
#    - blur(라플라시안 분산)
#    - low contrast(채널 std)
#    - under/over exposure(mean)
#    - 검은 테두리 비율(어두운 픽셀 비율)
# ============================================================
def step2_low_quality_candidates():
    rows = []

    for split in SPLITS:
        split_dir = f"{OUTPUT_BASE_DIR}/{split}"
        imgs = list_images(split_dir)

        for p in tqdm(imgs, desc=f"[STEP2] Quality scan {split}"):
            img_id = p.stem
            try:
                img = pil_open_verify(p)
                img = rgb_image(img)
                w, h = img.size

                rgb = np.array(img)
                gray = rgb.mean(axis=2).astype(np.uint8)

                # metrics
                m_mean = float(gray.mean())
                m_std = float(gray.std())
                m_lap = laplacian_var(gray)
                m_border = dark_border_ratio(rgb, thr=15)

                flags = []
                if m_lap < BLUR_LAPL_VAR_TH:
                    flags.append("blur_suspect")
                if m_std < LOW_CONTRAST_STD_TH:
                    flags.append("low_contrast")
                if m_mean < DARK_MEAN_TH:
                    flags.append("too_dark")
                if m_mean > BRIGHT_MEAN_TH:
                    flags.append("too_bright")
                if m_border > DARK_BORDER_RATIO_TH:
                    flags.append("large_black_border")

                rows.append({
                    "split": split, "img_id": img_id, "path": str(p),
                    "width": w, "height": h,
                    "mean_gray": round(m_mean, 3),
                    "std_gray": round(m_std, 3),
                    "lap_var": round(m_lap, 3),
                    "dark_border_ratio": round(m_border, 4),
                    "flags": "|".join(flags) if flags else ""
                })
                img.close()
            except Exception as e:
                # step1에서 이미 걸러졌을 수도 있지만, 안전하게 기록
                rows.append({
                    "split": split, "img_id": img_id, "path": str(p),
                    "width": None, "height": None,
                    "mean_gray": None, "std_gray": None, "lap_var": None, "dark_border_ratio": None,
                    "flags": "open_fail_in_quality_scan",
                })

    df = pd.DataFrame(rows)
    df["is_candidate"] = df["flags"].apply(lambda x: 1 if isinstance(x,str) and len(x)>0 else 0)

    out_all = f"{OUT_DIR}/step2_quality_metrics_all.csv"
    out_candidates = f"{OUT_DIR}/step2_low_quality_candidates.csv"
    df.to_csv(out_all, index=False)
    df[df["is_candidate"]==1].to_csv(out_candidates, index=False)

    print("\n✅ STEP2 저장 완료")
    print(" - all metrics:", out_all)
    print(" - candidates :", out_candidates)
    return df

# ============================================================
# E. 3) 중복 데이터 탐지
#    - exact dup: md5
#    - near dup: dHash + 해밍거리
#    * 기본은 split별로 돌리되, 누수 방지를 위해 전체 통합도 같이 가능
# ============================================================
def step3_duplicates():
    recs = []

    # 3-1) 해시 계산
    for split in SPLITS:
        split_dir = f"{OUTPUT_BASE_DIR}/{split}"
        imgs = list_images(split_dir)

        for p in tqdm(imgs, desc=f"[STEP3] Hashing {split}"):
            img_id = p.stem
            try:
                md5 = file_md5(p) if DO_EXACT_DUP else None
                phash = None
                if DO_NEAR_DUP:
                    img = pil_open_verify(p)
                    phash = dhash(img, hash_size=DHASH_SIZE)
                    img.close()

                recs.append({
                    "split": split, "img_id": img_id, "path": str(p),
                    "md5": md5, "dhash": phash
                })
            except Exception as e:
                recs.append({
                    "split": split, "img_id": img_id, "path": str(p),
                    "md5": None, "dhash": None, "error": str(e)
                })

    df = pd.DataFrame(recs)
    df.to_csv(f"{OUT_DIR}/step3_hashes.csv", index=False)

    # 3-2) exact duplicates
    exact_groups = []
    if DO_EXACT_DUP:
        g = df.dropna(subset=["md5"]).groupby("md5")
        for md5, sub in g:
            if len(sub) >= 2:
                exact_groups.append(sub.assign(dup_type="exact_md5", group_key=md5))
        exact_df = pd.concat(exact_groups, ignore_index=True) if exact_groups else pd.DataFrame()
        exact_df.to_csv(f"{OUT_DIR}/step3_exact_duplicates.csv", index=False)
    else:
        exact_df = pd.DataFrame()

    # 3-3) near duplicates (간단 O(N^2) → split별로만 수행)
    # 데이터가 매우 많으면 느릴 수 있어서 split별 제한 + 기준 조절
    near_pairs = []
    if DO_NEAR_DUP:
        for split in SPLITS:
            sub = df[(df["split"]==split) & df["dhash"].notna()].copy()
            hashes = sub["dhash"].tolist()
            paths = sub["path"].tolist()
            ids = sub["img_id"].tolist()

            n = len(hashes)
            # 너무 많으면 과부하 → 샘플링/블로킹 필요
            for i in tqdm(range(n), desc=f"[STEP3] Near-dup compare {split}"):
                for j in range(i+1, n):
                    d = hamming_hex(hashes[i], hashes[j])
                    if d <= NEAR_DUP_HAMMING_TH:
                        near_pairs.append({
                            "split": split,
                            "img_id_a": ids[i], "path_a": paths[i],
                            "img_id_b": ids[j], "path_b": paths[j],
                            "dhash_a": hashes[i], "dhash_b": hashes[j],
                            "hamming": d
                        })

    near_df = pd.DataFrame(near_pairs).sort_values(["split","hamming"], ascending=[True, True]) if len(near_pairs)>0 else pd.DataFrame()
    near_df.to_csv(f"{OUT_DIR}/step3_near_duplicates_pairs.csv", index=False)

    print("\n✅ STEP3 저장 완료")
    print(" - hashes:", f"{OUT_DIR}/step3_hashes.csv")
    if DO_EXACT_DUP:
        print(" - exact dup:", f"{OUT_DIR}/step3_exact_duplicates.csv")
    if DO_NEAR_DUP:
        print(" - near dup pairs:", f"{OUT_DIR}/step3_near_duplicates_pairs.csv")

    return exact_df, near_df

# ============================================================
# F. 실행 (요청한 순서대로)
# ============================================================
print("============================================================")
print("RUN OUTPUT_BASE_DIR =", OUTPUT_BASE_DIR)
print("RUN REPORT OUT_DIR  =", OUT_DIR)
print("QUARANTINE MOVE     =", ENABLE_QUARANTINE_MOVE)
print("============================================================")

# STEP 1
defects_df = step1_defect_scan()

# STEP 2
quality_df = step2_low_quality_candidates()

# STEP 3
exact_df, near_df = step3_duplicates()

print("\n🎉 전체 완료!")
print("결과 폴더:", OUT_DIR)


RUN OUTPUT_BASE_DIR = /content/drive/MyDrive/ColabNotebooks/RFMiD_A/Preprocessed
RUN REPORT OUT_DIR  = /content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/dataset_qc_run
QUARANTINE MOVE     = False
[labels:train] label csv not found -> 라벨 누락 탐지 스킵: /content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv/Training_Labels.csv
[labels:val] label csv not found -> 라벨 누락 탐지 스킵: /content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv/Validation_Labels.csv
[labels:test] label csv not found -> 라벨 누락 탐지 스킵: /content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv/Testing_Labels.csv


[STEP1] Defect scan train: 0it [00:00, ?it/s]
[STEP1] Defect scan val: 0it [00:00, ?it/s]
[STEP1] Defect scan test: 0it [00:00, ?it/s]



✅ STEP1 저장 완료
 - defects: /content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/dataset_qc_run/step1_defects.csv (rows=0)
 - summary: /content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/dataset_qc_run/step1_summary.csv


[STEP2] Quality scan train: 0it [00:00, ?it/s]
[STEP2] Quality scan val: 0it [00:00, ?it/s]
[STEP2] Quality scan test: 0it [00:00, ?it/s]


KeyError: 'flags'

step2) 제거 후보 데이터 조정("한 조건만 걸려도 후보"(or) -> "조합 조건"(AND/2개 이상)

In [ ]:
import pandas as pd

df = quality_df.copy()
df = df[df["flags"] != "open_fail_in_quality_scan"].copy()

# 분위수 임계값(원하는 비율로 조절)
q_blur = df["lap_var"].quantile(0.05)            # 하위 5%
q_contrast = df["std_gray"].quantile(0.05)       # 하위 5%
q_border = df["dark_border_ratio"].quantile(0.95)# 상위 5%
q_dark = df["mean_gray"].quantile(0.02)          # 하위 2%
q_bright = df["mean_gray"].quantile(0.98)        # 상위 2%

df["cand_blur"] = (df["lap_var"] <= q_blur).astype(int)
df["cand_low_contrast"] = (df["std_gray"] <= q_contrast).astype(int)
df["cand_border"] = (df["dark_border_ratio"] >= q_border).astype(int)
df["cand_dark"] = (df["mean_gray"] <= q_dark).astype(int)
df["cand_bright"] = (df["mean_gray"] >= q_bright).astype(int)

# 후보를 "OR"로 잡지 말고, 최소 2개 이상 조건 만족할 때만 후보로
df["cand_count"] = df[["cand_blur","cand_low_contrast","cand_border","cand_dark","cand_bright"]].sum(axis=1)
candidates = df[df["cand_count"] >= 2].copy()

print("임계값:")
print(" blur lap_var <= ", q_blur)
print(" contrast std <= ", q_contrast)
print(" border ratio >= ", q_border)
print(" dark mean <= ", q_dark)
print(" bright mean >= ", q_bright)
print("\n후보 수:", len(candidates), "/", len(df))

# 저장
candidates.to_csv(f"{OUT_DIR}/step2_low_quality_candidates_STRICT.csv", index=False)


step2) 제거 후보 CSV에 있는 이미지들 확인  후, 삭제

In [ ]:
#======================
# 실행 코드
#======================
import pandas as pd
from pathlib import Path

CAND_CSV = f"{OUT_DIR}"
MAX_SHOW = 50   # 너무 많으면 렉 걸려서 50~100 추천

cand = pd.read_csv(CAND_CSV)
cand = cand.head(MAX_SHOW).copy()
cand
#========================
# 이미지 갤러리 출력 코드
#========================
from IPython.display import display, HTML
import base64

def img_to_base64(img_path, max_width=260):
    img_path = str(img_path)
    with open(img_path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    ext = Path(img_path).suffix.lower().replace(".", "")
    return f"data:image/{ext};base64,{data}"

html = "<h3>Low-quality 후보 이미지 미리보기</h3>"
html += "<div style='display:flex; flex-wrap:wrap; gap:12px;'>"

for _, row in cand.iterrows():
    path = row["path"]
    info = f"""
    <div style="width:280px; border:1px solid #ddd; border-radius:10px; padding:8px;">
      <img src="{img_to_base64(path)}" style="width:100%; border-radius:8px;"/>
      <div style="font-size:12px; margin-top:6px;">
        <b>ID:</b> {row.get('img_id','')}<br/>
        <b>flags:</b> {row.get('flags','')}<br/>
        <b>lap_var:</b> {row.get('lap_var','')}<br/>
        <b>std:</b> {row.get('std_gray','')} | <b>mean:</b> {row.get('mean_gray','')}<br/>
        <b>border:</b> {row.get('dark_border_ratio','')}
      </div>
    </div>
    """
    html += info

html += "</div>"
display(HTML(html))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/dataset_qc_run/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/dataset_qc_run/step2_low_quality_candidates.csv'

step2) 제거 후보 이미지를 제외한 "Clean dataset"

In [ ]:
import os
import shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# =========================
# 1) 경로 설정
# =========================
# 원본 전처리 이미지 폴더
OUTPUT_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Preprocessed"

# 후보 CSV (step2에서 생성된 후보 파일)
CAND_CSV = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/dataset_qc_run/step2_low_quality_candidates_STRICT.csv"

# 후보 제외한 clean 데이터셋 저장 위치
CLEAN_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Clean_NoCandidates"

SPLITS = ["train", "val", "test"]
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# =========================
# 2) 후보 이미지 목록 로드
# =========================
cand = pd.read_csv(CAND_CSV)

# 후보 이미지들의 전체 path set
candidate_paths = set(cand["path"].astype(str).tolist())

print("✅ 후보 이미지 수:", len(candidate_paths))

# =========================
# 3) split별로 후보 제외 복사
# =========================
for split in SPLITS:
    src_dir = Path(OUTPUT_BASE_DIR) / split
    dst_dir = Path(CLEAN_BASE_DIR) / split
    os.makedirs(dst_dir, exist_ok=True)

    imgs = [p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]

    kept = 0
    removed = 0

    for p in tqdm(imgs, desc=f"Building CLEAN {split}"):
        if str(p) in candidate_paths:
            removed += 1
            continue

        shutil.copy(str(p), str(dst_dir / p.name))
        kept += 1

    print(f"\n[{split}] total={len(imgs)}  kept={kept}  removed(candidates)={removed}")

print("\n🎉 완료!")
print("✅ 후보 제외 CLEAN 데이터셋 생성 위치:", CLEAN_BASE_DIR)


step2) clean dataset에 제거 후보가잘 제거 되었는지 확인

In [ ]:
import pandas as pd
import os
from pathlib import Path

CAND_CSV = f"{OUT_DIR}/step2_low_quality_candidates_STRICT.csv"
CLEAN_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Clean_NoCandidates"

cand = pd.read_csv(CAND_CSV)
candidate_paths = set(cand["path"].astype(str).tolist())

# 후보 파일명만 추출
cand_names = set(Path(p).name for p in candidate_paths)

# Clean 폴더에 존재하는 파일명
clean_train = set(os.listdir(f"{CLEAN_BASE_DIR}/train"))

# 교집합 확인
remain = cand_names.intersection(clean_train)
print("🚨 Clean에 남아있는 후보 이미지 수:", len(remain))
print(list(remain)[:20])


In [ ]:
import pandas as pd
import os
from pathlib import Path

CAND_CSV = f"{OUT_DIR}/step2_low_quality_candidates_STRICT.csv"
CLEAN_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Clean_NoCandidates"

cand = pd.read_csv(CAND_CSV)

# (A) 후보를 "train 후보 파일명"으로만 제한해서 비교 (중요!)
cand_train_names = set(
    cand[cand["split"]=="train"]["path"].astype(str).apply(lambda x: Path(x).name).tolist()
)

clean_train_names = set(os.listdir(f"{CLEAN_BASE_DIR}/train"))

remain = sorted(list(cand_train_names.intersection(clean_train_names)))
print("🚨 train 후보인데 clean/train에 남아있는 파일 수:", len(remain))
print("예시:", remain[:20])

# (B) 남아있는 파일이 후보 CSV에 어떤 경로로 저장되어 있는지 확인(경로 불일치 진단)
if len(remain) > 0:
    sample = remain[:10]
    sub = cand[(cand["split"]=="train") & (cand["path"].astype(str).str.endswith(tuple(sample)))]
    print("\n[후보 CSV 경로 샘플]")
    print(sub[["img_id","path"]].head(10))

    # clean 폴더에 실제 존재하는 절대 경로도 출력
    print("\n[clean/train 실제 경로 샘플]")
    for fn in sample[:5]:
        print(str(Path(CLEAN_BASE_DIR)/"train"/fn))


step2) clean dataset 최종!

In [ ]:
import pandas as pd
import os
from pathlib import Path

CAND_CSV = f"{OUT_DIR}/step2_low_quality_candidates_STRICT.csv"
CLEAN_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Clean_NoCandidates"

cand = pd.read_csv(CAND_CSV)

def check_split(split):
    cand_names = set(cand[cand["split"]==split]["path"].astype(str).apply(lambda x: Path(x).name))
    clean_names = set(os.listdir(f"{CLEAN_BASE_DIR}/{split}"))
    remain = cand_names.intersection(clean_names)
    print(f"✅ {split}: 후보가 clean/{split}에 남아있는 수 =", len(remain))
    if len(remain) > 0:
        print("예시:", list(remain)[:20])

for sp in ["train","val","test"]:
    check_split(sp)


In [ ]:
clean 폴더 기준 라벨 csv 필터링 코드

In [ ]:
import os
from pathlib import Path
import pandas as pd

# =========================
# 0) 경로 설정
# =========================
CLEAN_BASE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Clean_NoCandidates"   # clean 이미지 폴더
CSV_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv"                         # 원본 라벨 csv 폴더
OUT_LABEL_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/csv_clean"             # clean 라벨 저장 폴더

LABEL_CSV_MAP = {
    "train": f"{CSV_DIR}/Training_Labels.csv",
    "val":   f"{CSV_DIR}/Validation_Labels.csv",
    "test":  f"{CSV_DIR}/Testing_Labels.csv",
}

SPLITS = ["train", "val", "test"]
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

os.makedirs(OUT_LABEL_DIR, exist_ok=True)

# =========================
# 1) 유틸: 라벨 CSV에서 ID 컬럼 자동 찾기
# =========================
def find_id_col(df: pd.DataFrame) -> str:
    for c in ["ID", "id", "image_id", "ImageID", "img_id", "filename"]:
        if c in df.columns:
            return c
    return df.columns[0]

def list_clean_ids(split: str):
    split_dir = Path(CLEAN_BASE_DIR) / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Clean split 폴더가 없음: {split_dir}")

    files = [p for p in split_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    # "617.png" -> "617"
    ids = set(p.stem for p in files)
    return ids, len(files)

# =========================
# 2) split별로 clean ids로 라벨 필터링
# =========================
all_clean_dfs = []
summary = []

for split in SPLITS:
    clean_ids, n_imgs = list_clean_ids(split)
    label_csv = LABEL_CSV_MAP[split]
    df = pd.read_csv(label_csv)

    id_col = find_id_col(df)
    df[id_col] = df[id_col].astype(str).str.strip()

    before = len(df)
    df_clean = df[df[id_col].isin(clean_ids)].copy()
    after = len(df_clean)

    out_path = f"{OUT_LABEL_DIR}/labels_clean_{split}.csv"
    df_clean.to_csv(out_path, index=False)

    all_clean_dfs.append(df_clean.assign(split=split))
    summary.append({
        "split": split,
        "clean_images_in_folder": n_imgs,
        "labels_before": before,
        "labels_after": after,
        "dropped_labels": before - after,
        "saved_to": out_path
    })

    print(f"✅ {split}: clean images={n_imgs} | labels {before} -> {after} | saved: {out_path}")

# =========================
# 3) 전체 합친 labels_clean_all.csv
# =========================
df_all = pd.concat(all_clean_dfs, ignore_index=True)
out_all = f"{OUT_LABEL_DIR}/labels_clean_all.csv"
df_all.to_csv(out_all, index=False)

print("\n🎉 전체 완료!")
print("✅ split별 저장 위치:", OUT_LABEL_DIR)
print("✅ 통합 파일:", out_all)

print("\n[요약]")
display(pd.DataFrame(summary))


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Clean_NoCandidates/csv_clean/labels_clean_test.csv")
print("행 개수:", len(df))          # 또는 df.shape[0]
print("열 개수:", df.shape[1])


# 라벨 분포 통계 정리

In [ ]:
import os
import pandas as pd

# =========================
# 0) 라벨 CSV 경로 설정 (clean 기준)
# =========================
LABEL_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Clean_NoCandidates/csv_clean"
LABEL_CSV_MAP = {
    "train": f"{LABEL_DIR}/labels_clean_train.csv",
    "val":   f"{LABEL_DIR}/labels_clean_val.csv",
    "test":  f"{LABEL_DIR}/labels_clean_test.csv",
}

OUT_REPORT_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/label_stats"
os.makedirs(OUT_REPORT_DIR, exist_ok=True)

# =========================
# 1) 유틸 함수
# =========================
def find_dr_col(df):
    if "DR" in df.columns:
        return "DR"
    for c in df.columns:
        if str(c).strip().lower() in ["dr", "dr_bin", "drbinary", "dr_binary"]:
            return c
    raise ValueError(f"DR 컬럼을 찾지 못했어. columns={list(df.columns)}")

def to_bin(series):
    """DR 값을 안전하게 0/1로 변환"""
    if series.dtype == object:
        series = pd.to_numeric(series, errors="coerce")
    series = series.fillna(0)
    return (series.astype(float) >= 1).astype(int)

def imbalance_level(pos_ratio: float) -> str:
    """
    imbalance 수준 기록(간단 룰)
    - very severe: pos <10% or >90%
    - severe     : pos <20% or >80%
    - moderate   : pos <35% or >65%
    - mild       : 그 외
    """
    if pos_ratio < 0.10 or pos_ratio > 0.90:
        return "very severe"
    elif pos_ratio < 0.20 or pos_ratio > 0.80:
        return "severe"
    elif pos_ratio < 0.35 or pos_ratio > 0.65:
        return "moderate"
    else:
        return "mild"

# =========================
# 2) split별 Binary 통계
# =========================
rows = []

for split, csv_path in LABEL_CSV_MAP.items():
    df = pd.read_csv(csv_path)

    dr_col = find_dr_col(df)
    dr_bin = to_bin(df[dr_col])

    total = len(df)
    pos = int(dr_bin.sum())
    neg = total - pos
    pos_ratio = (pos / total) if total > 0 else 0.0

    rows.append({
        "split": split,
        "total": total,
        "DR_pos(1)": pos,
        "DR_neg(0)": neg,
        "pos_ratio": round(pos_ratio, 4),
        "pos_to_neg": f"{pos}:{neg}" if neg != 0 else "inf",
        "imbalance_level": imbalance_level(pos_ratio)
    })

result_df = pd.DataFrame(rows)

# =========================
# 3) 저장 + 출력
# =========================
out_path = f"{OUT_REPORT_DIR}/binary_DR_stats.csv"
result_df.to_csv(out_path, index=False)

print("✅ Binary DR 통계 저장 완료:", out_path)
result_df


Binary(DR) 통계 그래프 코드

# PPT Figure 자동 생성 코드

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# =========================
# 0) 통계 CSV 불러오기
# =========================
STATS_CSV = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/label_stats/binary_DR_stats.csv"
df = pd.read_csv(STATS_CSV)

splits = df["split"].tolist()
pos = df["DR_pos(1)"].values.astype(int)
neg = df["DR_neg(0)"].values.astype(int)
total = df["total"].values.astype(int)

pos_pct = (pos / total) * 100
neg_pct = (neg / total) * 100

# 저장 경로
SAVE_DIR = "/content/drive/MyDrive/ColabNotebooks/RFMiD_A/Reports/label_stats/figures"
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# 스타일 세팅 (폰트/크기/팔레트/테두리/저장 dpi)
# =========================
PALETTE = ["#A6CEE3", "#B2DF8A", "#FDBF6F", "#CAB2D6", "#FB9A99"]  # 1~5번
FONT_FAMILY = "Times New Roman"
FONT_SIZE = 11
DPI = 600

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": FONT_SIZE,
    "axes.titlesize": FONT_SIZE + 1,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
})

def _beautify_axes(ax):
    # 테두리(스파인) 보이게 + 얇게
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
    # 축 눈금도 너무 두껍지 않게
    ax.tick_params(width=0.8)
    # 레이아웃 밀림 방지용
    ax.margins(x=0.05)

# =========================
# 1) Split별 DR 0/1 개수 (stacked bar) + % 표시
# =========================
x = np.arange(len(splits))
width = 0.6

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)  # 화면용 dpi는 가볍게, 저장은 600으로

# 색: neg = 1번, pos = 2번 (원하면 바꿔도 됨)
bar_neg = ax.bar(
    x, neg, width=width,
    label="DR=0 (neg)",
    color=PALETTE[0],
    edgecolor="black", linewidth=0.8
)
bar_pos = ax.bar(
    x, pos, width=width, bottom=neg,
    label="DR=1 (pos)",
    color=PALETTE[1],
    edgecolor="black", linewidth=0.8
)

ax.set_xticks(x, splits)
ax.set_ylabel("Count")
ax.set_xlabel("Split")
ax.set_title("DR label distribution by split (count + percent)")
ax.legend(frameon=True)

# % 텍스트 표시 (글씨가 겹칠 수 있으면 fontsize를 9~10으로만 낮추면 됨)
for i in range(len(splits)):
    # neg % (아래 영역 중앙)
    ax.text(
        x[i], neg[i] / 2,
        f"{neg_pct[i]:.1f}%",
        ha="center", va="center"
    )
    # pos % (위 영역 중앙)
    ax.text(
        x[i], neg[i] + pos[i] / 2,
        f"{pos_pct[i]:.1f}%",
        ha="center", va="center"
    )

_beautify_axes(ax)

out1 = os.path.join(SAVE_DIR, "dr_counts_stacked_with_percent.png")
fig.tight_layout()
fig.savefig(out1, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("✅ Saved:", out1)

# =========================
# 2) Split별 DR positive 비율 bar + % 표시
# =========================
ratio = df["pos_ratio"].values

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)

# 단일 바 색: 3번
bars = ax.bar(
    x, ratio, width=width,
    color=PALETTE[2],
    edgecolor="black", linewidth=0.8
)

ax.set_xticks(x, splits)
ax.set_ylim(0, 1)
ax.set_ylabel("Positive ratio")
ax.set_xlabel("Split")
ax.set_title("DR positive ratio by split")

# 위쪽 % 텍스트 (ymax=1이라 너무 위로 튀지 않도록 clamp)
for i, v in enumerate(ratio):
    y_text = min(v + 0.03, 0.98)
    ax.text(i, y_text, f"{v*100:.1f}%", ha="center", va="bottom")

_beautify_axes(ax)

out2 = os.path.join(SAVE_DIR, "dr_positive_ratio.png")
fig.tight_layout()
fig.savefig(out2, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("✅ Saved:", out2)

print("\n📌 저장 폴더:", SAVE_DIR)
